# Variant Analysis and Population Genomics
## TL;DR — Plain English Introduction

**What you will learn:** How to parse and analyze genetic variants (SNPs, indels) from VCF files, compute population genetics statistics, and visualize variant distributions.

**Why it matters:** Variant analysis underlies GWAS, pharmacogenomics, and precision medicine. Understanding allele frequencies, linkage disequilibrium, and population stratification is essential for anyone working with human genomic data.


## 🎯 Predict Before Running

1. A SNP has allele frequencies 0.6 (A) and 0.4 (G) in a population in Hardy-Weinberg equilibrium. What is the expected frequency of heterozygotes (AG)?
2. What does a Manhattan plot y-axis show? What does a p-value of 1×10⁻⁸ represent in GWAS?
3. What is linkage disequilibrium (LD)? If two SNPs are in perfect LD (r²=1), what does that tell you?


## 🟢 Complete Beginner's Guide

**Assumed background:** Zero genomics knowledge required.

### What you need to know before starting

- **SNP (Single Nucleotide Polymorphism)** — a single DNA base that differs between individuals (e.g. position 1000 on chromosome 7 is 'A' in most people but 'G' in some).
- **Allele** — one version of a gene or DNA position. At any SNP you have a reference allele (common) and an alternate allele (variant).
- **Genotype** — the pair of alleles you inherited (one from each parent). Written as AA, AG, or GG for a two-allele SNP.
- **Hardy-Weinberg Equilibrium (HWE)** — a mathematical prediction of genotype frequencies in a population with no selection, mutation, or drift. Deviations from HWE can signal genotyping errors or selection.
- **p-value** — probability that a result as extreme as observed would happen by chance. Low p-value (< 0.05) means statistically significant.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `VCF file` | Standard file format for storing variant calls (one row per variant) |
| `MAF (Minor Allele Frequency)` | How common the rarer allele is in the population |
| `GWAS` | Genome-Wide Association Study — testing millions of SNPs for disease links |
| `odds ratio` | Ratio of disease odds between allele carriers and non-carriers |
| `linkage disequilibrium` | Non-random association between alleles at nearby positions |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — understand what biological question is being asked.
2. **Run code cells one at a time** — print intermediate variables to see what they contain.
3. **Modify one number and re-run** — change the MAF filter threshold and see how many variants are kept.

### If you're stuck

- **YouTube:** 'What is a SNP?' by yourgenome (https://www.youtube.com/watch?v=3TIBFkPGGss) — 2-min intro.
- **YouTube:** 'StatQuest: Hardy-Weinberg Equilibrium' by Josh Starmer.
- **Book:** *Bioinformatics Algorithms* by Compeau & Pevzner, Chapter 5 (sequence variation).
- **Web:** https://www.ebi.ac.uk/training/online/courses/human-genetic-variation-introduction/


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import re

# ── SIMULATE A REALISTIC VCF FILE ──
np.random.seed(42)

n_variants = 1000
n_samples = 50

# Simulate a chromosome 22 segment (48-51 Mb)
positions = np.sort(np.random.randint(48_000_000, 51_000_000, n_variants))
ref_alleles = np.random.choice(['A', 'C', 'G', 'T'], n_variants)
alt_alleles = np.array([{'A':'T','C':'G','G':'A','T':'C'}[r] for r in ref_alleles])

# Random allele frequencies (realistic: most variants are rare)
# Use a beta distribution skewed toward low MAF
mafs = np.random.beta(0.5, 5, n_variants)  # Most variants < 10% MAF

# Generate genotypes for each sample
genotypes = np.zeros((n_variants, n_samples), dtype=int)  # 0=0/0, 1=0/1, 2=1/1
for v in range(n_variants):
    maf = mafs[v]
    # Under HWE: P(0/0)=(1-maf)^2, P(0/1)=2*maf*(1-maf), P(1/1)=maf^2
    probs = [(1-maf)**2, 2*maf*(1-maf), maf**2]
    genotypes[v] = np.random.choice([0, 1, 2], size=n_samples, p=probs)

# Build VCF-like DataFrame
sample_ids = [f'SAMPLE_{i:03d}' for i in range(n_samples)]
vcf_df = pd.DataFrame({
    'CHROM': 'chr22',
    'POS': positions,
    'ID': [f'rs{pos}' for pos in positions],
    'REF': ref_alleles,
    'ALT': alt_alleles,
    'QUAL': np.random.randint(20, 100, n_variants),
    'FILTER': 'PASS',
})
for i, s in enumerate(sample_ids):
    gt_map = {0: '0/0', 1: '0/1', 2: '1/1'}
    vcf_df[s] = [gt_map[g] for g in genotypes[:, i]]

print(f"Simulated VCF: {n_variants} variants, {n_samples} samples")
print(f"Region: chr22:48,000,000-51,000,000")
print()
print("VCF format (first 5 variants, first 5 samples):")
print(vcf_df[['CHROM','POS','REF','ALT','QUAL','FILTER'] + sample_ids[:3]].head())

In [ ]:
# ── ALLELE FREQUENCY SPECTRUM ──
# Key QC metric: distribution of minor allele frequencies

def compute_maf(genotype_row):
    """Compute minor allele frequency from genotype codes (0,1,2)."""
    gt = genotype_row.values
    n_alleles = 2 * len(gt)
    n_alt = gt.sum()  # 0/0=0, 0/1=1, 1/1=2 alt alleles
    freq = n_alt / n_alleles
    return min(freq, 1 - freq)  # MAF is always <= 0.5

# Compute MAF for all variants
geno_cols = sample_ids
observed_mafs = vcf_df[geno_cols].apply(
    lambda row: compute_maf(row.map({'0/0':0,'0/1':1,'1/1':2})), axis=1
)

# Hardy-Weinberg Equilibrium test for each variant
def hwi_test(genotype_row):
    """Chi-square test for HWE departure."""
    gt_counts = genotype_row.value_counts()
    n_00 = gt_counts.get('0/0', 0)
    n_01 = gt_counts.get('0/1', 0)
    n_11 = gt_counts.get('1/1', 0)
    n = n_00 + n_01 + n_11
    if n == 0: return 1.0
    p = (2*n_00 + n_01) / (2*n)
    q = 1 - p
    exp_00 = n * p**2
    exp_01 = n * 2*p*q
    exp_11 = n * q**2
    if exp_00 < 5 or exp_11 < 5: return 1.0  # too few
    from scipy.stats import chi2
    chi2_stat = ((n_00-exp_00)**2/exp_00 + (n_01-exp_01)**2/exp_01 +
                 (n_11-exp_11)**2/exp_11)
    return chi2.sf(chi2_stat, df=1)

hwe_pvals = vcf_df[geno_cols].apply(hwi_test, axis=1)
n_hwe_fail = (hwe_pvals < 1e-6).sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Minor Allele Frequency spectrum
axes[0].hist(observed_mafs, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Minor Allele Frequency (MAF)')
axes[0].set_ylabel('Number of variants')
axes[0].set_title('Allele Frequency Spectrum')
axes[0].axvline(0.01, color='red', linestyle='--', label='MAF=1% filter')
axes[0].axvline(0.05, color='orange', linestyle='--', label='MAF=5% filter')
axes[0].legend(fontsize=8)

# 2. HWE p-value distribution
axes[1].hist(np.log10(hwe_pvals + 1e-20), bins=50, color='coral', edgecolor='white', linewidth=0.5)
axes[1].axvline(-6, color='red', linestyle='--', label='p=1e-6 threshold')
axes[1].set_xlabel('log₁₀(HWE p-value)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'HWE Test Distribution\n({n_hwe_fail} variants fail HWE)')
axes[1].legend(fontsize=8)

# 3. QC: QUAL score vs MAF
sc = axes[2].scatter(observed_mafs, vcf_df['QUAL'], alpha=0.3, s=5, c='darkgreen')
axes[2].set_xlabel('Minor Allele Frequency')
axes[2].set_ylabel('Variant Quality Score')
axes[2].set_title('Quality vs MAF\n(rare variants often have lower quality)')

plt.tight_layout()
plt.savefig('variant_qc.png', dpi=100)
plt.show()

print(f"Variant QC Summary:")
print(f"  Total variants: {n_variants}")
print(f"  Monomorphic (MAF=0): {(observed_mafs==0).sum()}")
print(f"  Rare (MAF < 1%): {(observed_mafs < 0.01).sum()}")
print(f"  Common (MAF >= 5%): {(observed_mafs >= 0.05).sum()}")
print(f"  HWE failures (p<1e-6): {n_hwe_fail} (may indicate genotyping errors or selection)")

print()
print("ANSWER TO PREDICT Q1:")
maf_ex = 0.4
hetero_freq = 2 * maf_ex * (1 - maf_ex)
print(f"  HWE heterozygote frequency = 2pq = 2×0.6×0.4 = {hetero_freq}")

In [ ]:
# ── LINKAGE DISEQUILIBRIUM (LD) CALCULATION ──
# LD measures correlation between variants: r² and D'

def compute_ld_r2(geno1, geno2):
    """Compute r² (correlation) between two biallelic markers."""
    # Convert to dosage (0, 1, 2 copies of alt allele)
    x = geno1.map({'0/0':0,'0/1':1,'1/1':2}).values.astype(float)
    y = geno2.map({'0/0':0,'0/1':1,'1/1':2}).values.astype(float)

    # Remove missing
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    if len(x) < 10: return 0.0

    # r² = squared correlation
    r = np.corrcoef(x, y)[0, 1]
    return r**2

# Compute LD for a window of 50 consecutive variants near position 49,000,000
window_center = 49_000_000
window = vcf_df[(vcf_df['POS'] >= window_center) &
                (vcf_df['POS'] <= window_center + 500_000)].head(30)
n_win = len(window)

ld_matrix = np.zeros((n_win, n_win))
for i in range(n_win):
    for j in range(i, n_win):
        r2 = compute_ld_r2(window.iloc[i][geno_cols], window.iloc[j][geno_cols])
        ld_matrix[i, j] = ld_matrix[j, i] = r2

# Visualize LD matrix (triangular heatmap)
fig, ax = plt.subplots(figsize=(8, 7))
masked = np.triu(ld_matrix)
im = ax.imshow(masked, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='r² (LD)')
ax.set_title(f'Linkage Disequilibrium (r²) Matrix\n{n_win} variants in 500kb window on chr22')
ax.set_xlabel('Variant index')
ax.set_ylabel('Variant index')
plt.tight_layout()
plt.savefig('ld_heatmap.png', dpi=100)
plt.show()

# Find LD blocks (r² > 0.8 extended regions)
high_ld_pairs = [(i, j) for i in range(n_win) for j in range(i+1, n_win)
                 if ld_matrix[i,j] > 0.8]
print(f"High LD pairs (r²>0.8): {len(high_ld_pairs)} out of {n_win*(n_win-1)//2} possible")
print()
print("ANSWER TO PREDICT Q3:")
print("  Linkage Disequilibrium (LD) = non-random association between alleles at different loci.")
print("  If r²=1: knowing allele at SNP A perfectly predicts allele at SNP B.")
print("  Physical basis: nearby SNPs rarely have recombination between them.")
print("  Implication: in GWAS, you need only 1 representative SNP per LD block.")
print()
print("ANSWER TO PREDICT Q2:")
print("  Manhattan plot y-axis: -log₁₀(p-value)")
print("  p=1×10⁻⁸ → -log₁₀(1e-8) = 8 → genome-wide significance threshold")
print("  Chosen to control for ~1M independent tests in a typical GWAS array")

In [ ]:
# ── MINI-GWAS SIMULATION ──
# Associate variants with a simulated quantitative trait

np.random.seed(42)

# Simulate a quantitative phenotype (e.g., blood pressure)
# True causal variant: use variant at index 200
causal_idx = 200
beta_causal = 0.8  # effect size (SD units per allele)

dosages = vcf_df.iloc[causal_idx][geno_cols].map({'0/0':0,'0/1':1,'1/1':2}).values.astype(float)
phenotype = 120 + beta_causal * dosages + np.random.normal(0, 5, n_samples)  # mean=120 (BP)

# Run association test for all variants
from scipy.stats import pearsonr

assoc_results = []
for v in range(n_variants):
    dos = vcf_df.iloc[v][geno_cols].map({'0/0':0,'0/1':1,'1/1':2}).values.astype(float)
    if dos.std() < 0.01:  # monomorphic
        assoc_results.append({'variant_idx': v, 'pos': positions[v], 'pvalue': 1.0})
        continue
    r, pval = pearsonr(dos, phenotype)
    assoc_results.append({'variant_idx': v, 'pos': positions[v], 'pvalue': pval})

assoc_df = pd.DataFrame(assoc_results)

# Manhattan plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(assoc_df['pos'] / 1e6, -np.log10(assoc_df['pvalue'] + 1e-30),
           s=8, alpha=0.6, color='steelblue')
ax.axhline(-np.log10(5e-8), color='red', linestyle='--', linewidth=1.5,
           label='Genome-wide significance (p=5×10⁻⁸)')
ax.axvline(positions[causal_idx]/1e6, color='orange', alpha=0.5,
           label=f'True causal variant (idx={causal_idx})')
ax.set_xlabel('Position on chr22 (Mb)')
ax.set_ylabel('-log₁₀(p-value)')
ax.set_title('Mini-GWAS: Association with simulated quantitative trait')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('gwas_manhattan.png', dpi=100)
plt.show()

# Best hit
best = assoc_df.nsmallest(1, 'pvalue').iloc[0]
print(f"Top association: variant at position {int(best.pos):,}")
print(f"  p-value: {best.pvalue:.2e}")
print(f"  True causal variant at: {int(positions[causal_idx]):,}")
print(f"  Distance from true causal: {abs(int(best.pos) - int(positions[causal_idx])):,} bp")
print()
print("PRODUCTION WORKFLOW (what PLINK / REGENIE does):")
print("  1. Quality control: HWE, call rate, heterozygosity filters")
print("  2. Population stratification correction: include PCA components as covariates")
print("  3. Mixed model to account for relatedness (BOLT-LMM, SAIGE)")
print("  4. LD-based clumping to identify independent signals")
print("  5. Fine-mapping to narrow down causal variant candidates")

---
## 📚 Resources — Variant Analysis & Population Genetics

### University Courses (All Free)
| Course | Key Content |
|--------|-------------|
| **[MIT 6.047/6.878 Computational Biology](https://stellar.mit.edu/S/course/6/fa15/6.047/)** | Lectures 14-17: population genetics, GWAS, variant calling |
| **[Harvard MCB112](https://mcb112.org/)** | Statistical tests for variant associations; HWE derivations |
| **[Stanford Genetics HUMBIO 2A](https://explorecourses.stanford.edu/)** | Population genetics foundations |
| **[EMBL-EBI Variant Analysis Training](https://www.ebi.ac.uk/training/online/courses/human-genetic-variation-introduction/)** | Free online; practical VCF analysis |

### Essential Reading (Free)
- **[PLINK documentation](https://www.cog-genomics.org/plink/)** — standard GWAS tool; understanding its methodology teaches population genetics
- **[GATK Best Practices](https://gatk.broadinstitute.org/hc/en-us/articles/360035535912)** — Broad Institute's variant calling pipeline (industry standard)
- **[An Introduction to Population Genetics](https://global.oup.com/academic/product/an-introduction-to-population-genetics-9780878932764)** — free chapters at author's site; Hardy-Weinberg, LD, drift

### Key Concepts to Master
1. **VCF format**: CHROM, POS, REF, ALT, QUAL, FILTER, INFO, FORMAT fields
2. **Hardy-Weinberg Equilibrium**: test for genotyping errors; deviations indicate selection or population structure
3. **Linkage Disequilibrium (r²)**: correlated alleles across loci; explains why GWAS signals span regions
4. **Manhattan plots**: -log10(p-value) vs genomic position; know the significance threshold (5×10⁻⁸)
5. **Allele frequency spectrum**: shape reveals population history (bottleneck, selection)

### Real Datasets (Free)
- **[1000 Genomes Project](https://www.internationalgenome.org/)** — 2504 individuals, 26 populations; standard reference
- **[UK Biobank](https://www.ukbiobank.ac.uk/)** — 500K individuals; requires application but widely used
- **[ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/)** — clinical significance of variants; free

### Statistical Tests You Must Know
| Test | Use Case | Key Assumption |
|------|----------|---------------|
| Chi-square | HWE, allele frequency difference | Expected counts > 5 |
| Fisher's exact | Case-control with rare variants | None (exact test) |
| Logistic regression | GWAS with covariates | Large sample |
| Linear mixed models | GWAS with population structure | MVN residuals |


---
## 🎯 Interview Questions — Variants & Population Genetics

**Q1:** What does Hardy-Weinberg Equilibrium tell us when it's violated in a GWAS dataset?
> **A:** HWE violation in controls suggests genotyping error (the most common cause), population stratification, or strong selection on that locus. In practice, variants failing HWE at p < 1e-6 in controls are filtered from GWAS because genotyping errors confound association signals.

**Q2:** What is linkage disequilibrium and why does it mean GWAS hits don't identify causal variants?
> **A:** LD is the non-random association between alleles at different loci — nearby variants tend to be inherited together (on the same haplotype). A GWAS hit at rs12345 may not be causal; it may just be in LD (r² ≈ 1) with the true causal variant rs99999 nearby. Fine-mapping and functional annotation are needed to identify the causal variant.

**Q3 (Drug discovery angle):** How would you use population-level variant data to identify drug targets?
> **A:** Human Genetic Evidence (HGE) strategy: find variants that affect gene expression (eQTLs) or protein function AND associate with disease. Genes with loss-of-function variants protecting against disease are drug targets (e.g., PCSK9 loss-of-function → lower LDL → statin target). This is now standard practice at Isomorphic Labs and pharma companies.


---
## 🔗 Cross-Module Connections

This notebook connects to:
- **Module 02/01 genomics_core (VCF format intro)**
- **Module 04/01 ml_for_omics (ML on variant features)**
- **Module 07/01 af3_architecture (variant effect on protein structure)**


**Progression:** After mastering this notebook, proceed to `04/01_ml_for_omics.ipynb`
to see how these features feed into ML models.


---
## ✅ Mastery Check — Variant Analysis

Before moving on, you should be able to answer these without looking:

1. Explain Hardy-Weinberg Equilibrium and what deviations indicate
2. Compute allele frequency from a VCF file with 5 samples
3. Explain why GWAS significance threshold is 5×10⁻⁸
4. Describe linkage disequilibrium in plain English
5. Design a GWAS analysis workflow: from raw VCF to significant hits


**Score yourself:**
- 5/5: You're ready to move on
- 3-4/5: Review the cells you couldn't answer, then re-attempt
- < 3/5: Re-read the MIT 7.91J lecture on this topic, then redo all cells

**Time estimate:** If these questions take > 10 minutes each, revisit the notebook.
At interview pace, you should answer in 2-3 minutes per question.


## 🐛 Debug Exercise — Hardy-Weinberg Equilibrium & GWAS

Find and fix the **3 bugs** in the variant analysis code below.

**Expected correct output:**
```
Allele freq p (A): 0.650
HWE chi2 p-value: > 0.05  (variant is in HWE)
GWAS variants tested (after MAF filter): < 1000
```

<details>
<summary>Show answer</summary>

**Bug 1 — Wrong degrees of freedom:** For a biallelic SNP the HWE chi-square test has **df=1**,
not df=2. Fix: `chi2.sf(stat, df=1)`.

**Bug 2 — Allele frequency from genotype counts:** `p = counts['AA'] / total` counts genotypes,
not alleles. Fix: `p = (2*counts['AA'] + counts['Aa']) / (2 * total)`.

**Bug 3 — No MAF filter before GWAS:** Running association tests on very rare variants inflates
false positives. Fix: filter out SNPs with MAF < 0.05 before the loop.

```python
maf_mask = np.minimum(af, 1 - af) >= 0.05
snp_matrix = snp_matrix[:, maf_mask]
```
</details>


In [ ]:
# DEBUG EXERCISE — HWE test and GWAS pipeline, find and fix 3 bugs
import numpy as np
from scipy.stats import chi2

# --- Hardy-Weinberg test ---
genotype_counts = {'AA': 298, 'Aa': 204, 'aa': 98}   # sample of 600 individuals
total = sum(genotype_counts.values())

# BUG 2: dividing AA count by total gives genotype freq, not allele freq
# correct formula: p = (2*AA + Aa) / (2 * total)
p = genotype_counts['AA'] / total          # WRONG — this is f(AA), not f(A)
q = 1 - p
print(f"Allele freq p (A): {p:.3f}")       # should be ~0.650

expected = {
    'AA': p**2 * total,
    'Aa': 2*p*q * total,
    'aa': q**2 * total,
}
chi2_stat = sum((genotype_counts[g] - expected[g])**2 / expected[g]
                for g in genotype_counts)

# BUG 1: biallelic SNP HWE test has df=1, not df=2
p_val = chi2.sf(chi2_stat, df=2)          # WRONG df
print(f"HWE chi2 p-value: {p_val:.4f}")   # should be > 0.05

# --- Simulated GWAS ---
np.random.seed(42)
n_samples, n_snps = 200, 1000
snp_matrix = np.random.binomial(2, 0.05, size=(n_samples, n_snps))  # many rare variants
phenotype  = np.random.randn(n_samples)

# BUG 3: no MAF filter — rare variants (MAF < 0.05) inflate false positives
# fix: compute allele frequencies and filter before running the loop
# af = snp_matrix.mean(axis=0) / 2
# maf_mask = np.minimum(af, 1 - af) >= 0.05
# snp_matrix = snp_matrix[:, maf_mask]

from scipy.stats import pearsonr
pvals = [pearsonr(snp_matrix[:, i], phenotype)[1] for i in range(snp_matrix.shape[1])]
sig = sum(p < 0.05 for p in pvals)
print(f"GWAS variants tested: {snp_matrix.shape[1]}, spuriously significant: {sig}")
